In [ ]:
import functools
import warnings

import botocore
import boto3
from iterpop import iterpop as ip
from matplotlib.markers import MarkerStyle
from matplotlib.ticker import MultipleLocator
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

from dishpylib.pyhelpers import fit_control_t_distns

warnings.filterwarnings("ignore")


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-08-13-eco-gwas-weakstrong"


In [ ]:
@functools.lru_cache
def get_control_t_distns( bucket, prefix, suffix, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/{prefix}control-competitions-lowestroot{suffix}/stage={4 + bool(prefix)}+what=collated/stint={stint}/',
    )

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )

    return fit_control_t_distns(control_df[
        control_df["Root ID"].isin([0, 1])
    ].copy())


In [ ]:
def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    print(len(competitions_df), "competitions to preprocess")
    print(len(control_fits_df), "control fits available")
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: scipy_stats.t.cdf(
            row["Fitness Differential"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 40
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 40)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
bucket = "prq49-20stint"
dfs = []
for suffix in [
    "-backgroundbb",
    "-focalbb",

]:
    prefix = ""
    if "step" in bucket:
        step = int(bucket.split("-step")[-1]) + 1
    else:
        step = 0
    if "restint" in bucket:
        kind = bucket.split("-")[3]
    else:
        kind = None
    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    # https://github.com/mmore500/oee4/blob/4cc2305ffdf41b5335875e9074e17773af47e734/binder/2026-07-25-ecoselfcontext-pgcomplex-screen.ipynb
    stint = 0
    for endeavor in tqdm([17, *range(20, 29)]):
        try:
            series_profiles, = bucket_handle.objects.filter(
                Prefix=f'endeavor={endeavor}/{prefix}variant-competitions-lowestroot{suffix}/stage=4+what=collated/stint={stint}/',
            )
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
            df = pd.read_csv(
                f's3://{bucket}/{series_profiles.key}',
                compression='xz',
            )
            df["Stint"] = stint
            df["Series"] = df["Competition Series"]
            dfdigest = "{:x}".format( hash_pandas_object( df ).sum() )
            assert "Series" in df.columns, df.columns

            df = df.copy()
            df["bucket"] = bucket
            df["variant"] = {"cryptic-": "skeleton", "": "wildtype"}[prefix]
            df["how"] = suffix.strip("-") if suffix else "none"
            dfs.append(df)
        except Exception as e:
            print(e)
            print(f"Skipping {bucket=}, {stint=}, {prefix=}, {suffix=}, {endeavor=}")
            raise e

print(len(dfs), "dataframes loaded")


In [ ]:
df = pd.concat(dfs)


In [ ]:
classifications = {
    24017: "mild",
    27030: "mild",
    26032: "mild",
    22011: "mild",
    26025: "mild",
    20027: "mild",
    24036: "mild",
    23034: "strong",
    23018: "strong",
    21038: "strong",
    27014: "strong",
    25027: "strong",
    20007: "strong",
    23025: "strong",
    # 17020: "strong",
    # 23039: "strong",
    # 24010: "strong",
}


In [ ]:
df = df[df["Series"].isin(classifications.keys())].copy()
df["classification"] = df["Series"].map(classifications)


In [ ]:
pd.options.display.max_columns = None


In [ ]:
dfx = df[df["Root ID"] == 1]
dfx["site"] = dfx["genome variation"].str.extract(r'i(\d+)%')


In [ ]:
for ythresh in [0.45, 0.5]:

    # 1. Pivot the data
    dat = dfx.pivot(
        index=["Series", "classification", "site"],
        columns=["how"],
        values=["Focal Prevalence"],
    ).reset_index()

    # 2. FLATTEN the MultiIndex columns
    dat.columns = [
        f"{col[0]}_{col[1]}".strip("_") if isinstance(col, tuple) else col
        for col in dat.columns
    ]

    # dat = dat[
    #     dat["classification"].isin(["best eco", "least eco"])
    # ]

    # Extract unique classes to force uniform colors/markers across all layers
    classes = dat["classification"].unique()

    # 3. Base plot with low alpha
    g = sns.relplot(
        data=dat,
        x="Focal Prevalence_backgroundbb",
        y="Focal Prevalence_focalbb",
        hue="classification",
        style="classification",
        alpha=0.18,
        col="classification",
        hue_order=classes,
        style_order=classes,
        legend=False,
        s=10,
    )

    thresh = 0.2

    # 4. Custom function to filter and plot points
    def plot_high_alpha(data, x, y, hue, style, **kwargs):
        # map_dataframe automatically passes 'color' and 'label' to kwargs.
        # We must remove them so they don't clash with sns.scatterplot's internal hue mapping.
        kwargs.pop("color", None)
        kwargs.pop("label", None)

        subset = data[(data[x] < thresh) & (data[y] >= ythresh)]
        if not subset.empty:
            sns.scatterplot(
                data=subset,
                x=x,
                y=y,
                hue=hue,
                style=style,
                **kwargs
            )

    # 5. Map the custom function
    g.map_dataframe(
        plot_high_alpha,
        x="Focal Prevalence_backgroundbb",
        y="Focal Prevalence_focalbb",
        hue="classification",
        style="classification",
        hue_order=classes,
        style_order=classes,
        alpha=0.8,     # High alpha for emphasis
        legend=False,  # Prevent duplicate entries in the legend
        zorder=5,       # Ensure these points draw on top of the low-alpha ones
        s=15,
    )

    # 6. Add custom lines
    for ax in g.axes.flatten():
        ax.axhline(0.5, ls="--", color="black", lw=1, zorder=-1)
        ax.axvline(0.5, ls="--", color="black", lw=1, zorder=-1)
        ax.plot((thresh, thresh), (1.0, ythresh), ls="--", color="red", lw=1, zorder=4)
        ax.plot((0, thresh), (ythresh, ythresh), ls="--", color="red", lw=1, zorder=4)

    g.figure.set_size_inches(4.5, 2)


In [ ]:
# 1. Define your condition
condition = (dat["Focal Prevalence_backgroundbb"] < 0.2) & (dat["Focal Prevalence_focalbb"] >= 0.45)

# 2. Assign as a temporary column, group the whole dataframe, and sum
result = (
    dat.assign(meets_condition=condition)
    .groupby(["classification", "Series"])["meets_condition"]
    .sum()
    .astype(int) # Converts the sum back to standard integers
    .reset_index(name="count")
)

sns.swarmplot(
    data=result,
    x="classification",
    y="count",
    # notch=True,
)

result, scipy_stats.mannwhitneyu(
    *[
        group["count"].values
        for name, group in result.groupby("classification")
    ]
)
